In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Fine-Tune BLIP ITM on VisualNews (NewsClipPings)

Fine-tunes `Salesforce/blip-itm-base-coco` on 50,000 balanced samples (25k real + 25k fake).
- Batch size 16, LR 1e-5, early stopping (patience 3) on validation F1
- Validates every epoch on 5,000 samples from val.json
- Saves the best checkpoint by validation F1

In [1]:
import json
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import BlipProcessor, BlipForImageTextRetrieval
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm import tqdm
import copy

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB" if hasattr(torch.cuda.get_device_properties(0), 'total_mem') else f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.0 GB


In [2]:
# ── Config ──
DATASET_ROOT = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
TRAIN_ANN_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
VAL_ANN_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "val.json")
TRAIN_META_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")
VAL_META_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "val.json")
IMAGE_BASE = os.path.join(DATASET_ROOT, "origin")  # visual_news/origin/X -> dataset/origin/origin/X

MODEL_NAME = "Salesforce/blip-itm-base-coco"
BATCH_SIZE = 16
LR = 1e-5
MAX_EPOCHS = 20
PATIENCE = 3
NUM_TRAIN = 50000  # 25k real + 25k fake
NUM_VAL = 5000     # 2.5k real + 2.5k fake
SAVE_DIR = _os.path.join(str(_cfg.ROOT), 'models', 'blip_itm_finetuned')
os.makedirs(SAVE_DIR, exist_ok=True)

def resolve_image_path(meta_image_path: str) -> str:
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGE_BASE, rel)

print("Config ready.")

Config ready.


In [3]:
# ── Load and prepare samples ──
def load_samples(ann_path, meta_path, num_per_class):
    """Load balanced samples (num_per_class real + num_per_class fake)."""
    with open(ann_path, "r", encoding="utf-8") as f:
        annotations = json.load(f)["annotations"]
    with open(meta_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    real_samples, fake_samples = [], []
    random.shuffle(annotations)

    for ann in annotations:
        if len(real_samples) >= num_per_class and len(fake_samples) >= num_per_class:
            break

        art_id = str(ann["id"])
        img_id = str(ann["image_id"])

        if art_id not in metadata or img_id not in metadata:
            continue

        img_path = resolve_image_path(metadata[img_id]["image_path"])
        if not os.path.isfile(img_path):
            continue

        caption = metadata[art_id]["caption"]
        label = 0 if not ann["falsified"] else 1  # 0=real, 1=fake

        entry = {"caption": caption, "image_path": img_path, "label": label}

        if label == 0 and len(real_samples) < num_per_class:
            real_samples.append(entry)
        elif label == 1 and len(fake_samples) < num_per_class:
            fake_samples.append(entry)

    samples = real_samples + fake_samples
    random.shuffle(samples)
    print(f"  Loaded {len(real_samples)} real + {len(fake_samples)} fake = {len(samples)} samples")
    return samples

print("Loading training samples...")
train_samples = load_samples(TRAIN_ANN_PATH, TRAIN_META_PATH, NUM_TRAIN // 2)

print("Loading validation samples...")
val_samples = load_samples(VAL_ANN_PATH, VAL_META_PATH, NUM_VAL // 2)

Loading training samples...
  Loaded 25000 real + 25000 fake = 50000 samples
Loading validation samples...
  Loaded 2500 real + 2500 fake = 5000 samples


In [4]:
# ── Check if val metadata is separate or shared ──
# If val metadata doesn't exist separately, try using train metadata for val
if not os.path.exists(VAL_META_PATH):
    print("Val metadata not found separately, checking alternatives...")
    # Try the origin data.json which has all articles
    GLOBAL_META_PATH = os.path.join(DATASET_ROOT, "origin", "origin", "data.json")
    if os.path.exists(GLOBAL_META_PATH):
        print(f"Using global metadata: {GLOBAL_META_PATH}")
else:
    print(f"Val metadata exists at: {VAL_META_PATH}")

Val metadata exists at: D:\Pics Can Lie\dataset\data\NewsClipPings\metadata\val.json


In [5]:
# ── Dataset class ──
class NewsClipPingsDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s["image_path"]).convert("RGB")
        inputs = self.processor(images=image, text=s["caption"], return_tensors="pt",
                                padding="max_length", truncation=True, max_length=77)
        # Squeeze batch dim added by processor
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        # Label: for ITM head, 1 = match (real), 0 = no match (fake)
        # Our label convention: 0=real, 1=fake
        # ITM head convention: class 1 = match, class 0 = no match
        # So ITM label = 1 - our_label (real->1, fake->0)
        item["labels"] = torch.tensor(1 - s["label"], dtype=torch.long)
        return item

print("Loading processor...")
processor = BlipProcessor.from_pretrained(MODEL_NAME)

train_dataset = NewsClipPingsDataset(train_samples, processor)
val_dataset = NewsClipPingsDataset(val_samples, processor)

# DEBUG: Verify dataset returns a valid item
print("[DEBUG] Testing single dataset item...")
test_item = train_dataset[0]
print(f"[DEBUG] Keys: {list(test_item.keys())}")
for k, v in test_item.items():
    print(f"[DEBUG]   {k}: shape={v.shape}, dtype={v.dtype}")
print("[DEBUG] Single item OK.")

# num_workers=0 to avoid Windows multiprocessing hang
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)

# DEBUG: Verify first batch loads
print("[DEBUG] Testing first batch...")
first_batch = next(iter(train_loader))
print(f"[DEBUG] Batch keys: {list(first_batch.keys())}")
for k, v in first_batch.items():
    print(f"[DEBUG]   {k}: shape={v.shape}, dtype={v.dtype}")
print("[DEBUG] First batch OK.")

print(f"\nTrain: {len(train_dataset)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} samples, {len(val_loader)} batches")

Loading processor...


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[DEBUG] Testing single dataset item...
[DEBUG] Keys: ['pixel_values', 'input_ids', 'attention_mask', 'labels']
[DEBUG]   pixel_values: shape=torch.Size([3, 384, 384]), dtype=torch.float32
[DEBUG]   input_ids: shape=torch.Size([77]), dtype=torch.int64
[DEBUG]   attention_mask: shape=torch.Size([77]), dtype=torch.int64
[DEBUG]   labels: shape=torch.Size([]), dtype=torch.int64
[DEBUG] Single item OK.
[DEBUG] Testing first batch...
[DEBUG] Batch keys: ['pixel_values', 'input_ids', 'attention_mask', 'labels']
[DEBUG]   pixel_values: shape=torch.Size([16, 3, 384, 384]), dtype=torch.float32
[DEBUG]   input_ids: shape=torch.Size([16, 77]), dtype=torch.int64
[DEBUG]   attention_mask: shape=torch.Size([16, 77]), dtype=torch.int64
[DEBUG]   labels: shape=torch.Size([16]), dtype=torch.int64
[DEBUG] First batch OK.

Train: 50000 samples, 3125 batches
Val:   5000 samples, 313 batches


In [6]:
# ── Load model & selective freeze ──
print(f"Loading {MODEL_NAME}...")
model = BlipForImageTextRetrieval.from_pretrained(MODEL_NAME).to(device)

# Step 1: Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze ITM head + projection layers
for param in model.itm_head.parameters():
    param.requires_grad = True
for param in model.vision_proj.parameters():
    param.requires_grad = True
for param in model.text_proj.parameters():
    param.requires_grad = True

# Step 3: Unfreeze last 2 layers of vision encoder (layers 10, 11)
for param in model.vision_model.encoder.layers[10].parameters():
    param.requires_grad = True
for param in model.vision_model.encoder.layers[11].parameters():
    param.requires_grad = True

# Step 4: Unfreeze last 2 layers of text encoder (layers 10, 11)
for param in model.text_encoder.encoder.layer[10].parameters():
    param.requires_grad = True
for param in model.text_encoder.encoder.layer[11].parameters():
    param.requires_grad = True

model.train()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
frozen_params = total_params - trainable_params
print(f"Total: {total_params:.1f}M | Frozen: {frozen_params:.1f}M | Trainable: {trainable_params:.1f}M")

# Show what's trainable
print("\nTrainable modules:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.numel():,}")

print(f"\n[DEBUG] Model device: {next(model.parameters()).device}")
print(f"[DEBUG] CUDA memory allocated: {torch.cuda.memory_allocated()/1024**2:.0f} MB")

Loading Salesforce/blip-itm-base-coco...


Loading weights: 100%|██████████| 472/472 [00:00<00:00, 18792.65it/s]
BlipForImageTextRetrieval LOAD REPORT from: Salesforce/blip-itm-base-coco
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_encoder.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total: 223.7M | Frozen: 190.3M | Trainable: 33.5M

Trainable modules:
  vision_model.encoder.layers.10.self_attn.qkv.weight: 1,769,472
  vision_model.encoder.layers.10.self_attn.qkv.bias: 2,304
  vision_model.encoder.layers.10.self_attn.projection.weight: 589,824
  vision_model.encoder.layers.10.self_attn.projection.bias: 768
  vision_model.encoder.layers.10.layer_norm1.weight: 768
  vision_model.encoder.layers.10.layer_norm1.bias: 768
  vision_model.encoder.layers.10.mlp.fc1.weight: 2,359,296
  vision_model.encoder.layers.10.mlp.fc1.bias: 3,072
  vision_model.encoder.layers.10.mlp.fc2.weight: 2,359,296
  vision_model.encoder.layers.10.mlp.fc2.bias: 768
  vision_model.encoder.layers.10.layer_norm2.weight: 768
  vision_model.encoder.layers.10.layer_norm2.bias: 768
  vision_model.encoder.layers.11.self_attn.qkv.weight: 1,769,472
  vision_model.encoder.layers.11.self_attn.qkv.bias: 2,304
  vision_model.encoder.layers.11.self_attn.projection.weight: 589,824
  vision_model.encoder.layers.11

In [7]:
# ── Training setup with differential learning rates ──
# ITM head + projections: LR 1e-5 (higher — these need to adapt most)
# Encoder layers 10-11:   LR 1e-6 (lower — gentle fine-tuning)

head_params = []
encoder_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "itm_head" in name or "vision_proj" in name or "text_proj" in name:
        head_params.append(param)
    else:
        encoder_params.append(param)

print(f"Head param group:    {sum(p.numel() for p in head_params):,} params @ LR=1e-5")
print(f"Encoder param group: {sum(p.numel() for p in encoder_params):,} params @ LR=1e-6")

optimizer = torch.optim.AdamW([
    {"params": head_params,    "lr": 1e-5},
    {"params": encoder_params, "lr": 1e-6},
], weight_decay=0.01)

criterion = nn.CrossEntropyLoss()

# Scheduler: linear warmup + cosine decay
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = len(train_loader)  # 1 epoch warmup
from transformers import get_cosine_schedule_with_warmup
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

print(f"Total steps: {total_steps}, Warmup: {warmup_steps}")

Head param group:    395,266 params @ LR=1e-5
Encoder param group: 33,079,296 params @ LR=1e-6
Total steps: 62500, Warmup: 3125


In [8]:
# ── Training loop with early stopping + auto-save ──
best_f1 = 0.0
best_epoch = 0
patience_counter = 0
history = {"train_loss": [], "val_f1": [], "val_acc": []}

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val F1':>8} | {'Val Acc':>8} | {'Status'}")
print("-" * 60)

for epoch in range(1, MAX_EPOCHS + 1):
    # ── Train ──
    model.train()
    running_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]", leave=False)
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")

        optimizer.zero_grad()
        outputs = model(**batch, use_itm_head=True)
        loss = criterion(outputs.itm_score, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / num_batches
    history["train_loss"].append(avg_loss)

    # ── Validate ──
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch} [Val]", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop("labels")

            outputs = model(**batch, use_itm_head=True)
            preds = outputs.itm_score.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ITM labels: 1=match(real), 0=no-match(fake)
    # For F1 on fake detection: convert to our convention (fake=1)
    y_true = [1 - l for l in all_labels]  # flip: ITM 1->0(real), ITM 0->1(fake)
    y_pred = [1 - p for p in all_preds]

    val_f1 = f1_score(y_true, y_pred)
    val_acc = accuracy_score(y_true, y_pred)
    history["val_f1"].append(val_f1)
    history["val_acc"].append(val_acc)

    # ── Early stopping + auto-save to disk ──
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0
        # Save directly to disk instead of keeping in RAM
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
            json.dump(history, f, indent=2)
        status = f"** BEST ** (saved to {SAVE_DIR})"
    else:
        patience_counter += 1
        status = f"patience {patience_counter}/{PATIENCE}"

    print(f"{epoch:>5} | {avg_loss:>10.4f} | {val_f1:>8.4f} | {val_acc:>8.4f} | {status}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch} (F1: {best_f1:.4f})")
        break

print(f"\nTraining complete. Best val F1: {best_f1:.4f} at epoch {best_epoch}")
print(f"Best checkpoint saved at: {SAVE_DIR}")

Epoch | Train Loss |   Val F1 |  Val Acc | Status
------------------------------------------------------------


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]                


    1 |     0.6545 |   0.7228 |   0.7300 | ** BEST ** (saved to D:\Pics Can Lie\blip_itm_finetuned)


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.97s/it]                


    2 |     0.5102 |   0.7307 |   0.7378 | ** BEST ** (saved to D:\Pics Can Lie\blip_itm_finetuned)


...

    8 |     0.4448 |   0.7457 |   0.7480 | patience 1/3


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.20s/it]                


    9 |     0.4374 |   0.7511 |   0.7494 | ** BEST ** (saved to D:\Pics Can Lie\blip_itm_finetuned)


   10 |     0.4310 |   0.7450 |   0.7484 | patience 1/3


KeyboardInterrupt: 

In [ ]:
# ── Verify saved checkpoint ──
import os
saved_files = os.listdir(SAVE_DIR)
print(f"Checkpoint files in {SAVE_DIR}:")
for f in saved_files:
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f} ({size / 1024 / 1024:.1f} MB)" if size > 1024*1024 else f"  {f} ({size / 1024:.1f} KB)")
print(f"\nBest checkpoint was auto-saved during training at epoch {best_epoch} (F1: {best_f1:.4f})")

In [ ]:
# ── Final validation report on best model ──
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Final eval"):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        outputs = model(**batch, use_itm_head=True)
        preds = outputs.itm_score.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_true = [1 - l for l in all_labels]
y_pred = [1 - p for p in all_preds]

print("\n" + "=" * 60)
print("BEST MODEL — VALIDATION RESULTS")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=["REAL", "FAKE"]))

In [ ]:
# ── Plot training curves ──
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Training loss
ax1.plot(epochs, history["train_loss"], "b-o", label="Train Loss", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Validation F1
ax2.plot(epochs, history["val_f1"], "r-o", label="Val F1", linewidth=2)
ax2.plot(epochs, history["val_acc"], "g--s", label="Val Accuracy", linewidth=2, alpha=0.7)
ax2.axvline(x=best_epoch, color="orange", linestyle="--", label=f"Best (epoch {best_epoch})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Validation Metrics")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f"BLIP ITM Fine-Tuning — Best Val F1: {best_f1:.4f} (Epoch {best_epoch})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(_os.path.join(str(_cfg.ROOT), 'results', 'blip_itm_finetune_results.png'), dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to blip_itm_finetune_results.png")